# Data inspection and cleaning

This notebook inspects the marketing A/B testing dataset without changing or exporting the original CSV.

In [ ]:
from pathlib import Path

import pandas as pd

candidate_paths = [
    Path.cwd() / "data" / "raw" / "marketing_AB.csv",
    Path.cwd().parent / "data" / "raw" / "marketing_AB.csv",
]
csv_path = next((path for path in candidate_paths if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Download marketing_AB.csv into data/raw/ and run from the project root or notebooks folder."
    )
df = pd.read_csv(csv_path)

print(f"Loaded: {csv_path}")
df.head()

In [ ]:
print("Shape:", df.shape)

print("\nColumn names and data types:")
df.dtypes.to_frame(name="data_type")

In [ ]:
print("Missing values by column:")
df.isna().sum().to_frame(name="missing_values")

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate user IDs:", df["user id"].duplicated().sum())

In [ ]:
print("Test group distribution:")
display(df["test group"].value_counts(dropna=False).to_frame(name="count"))

print("Converted distribution:")
display(df["converted"].value_counts(dropna=False).to_frame(name="count"))

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "str", "bool", "category"]).columns
print("Unique values in categorical columns:")
for column in categorical_columns:
    print(f"\n{column}: {df[column].dropna().unique().tolist()}")

In [ ]:
unnecessary_columns = [column for column in df.columns if column.startswith("Unnamed:")]
print("Potentially unnecessary columns:", unnecessary_columns)
print("\nPotentially suspicious values:")
print("Unexpected test groups:", sorted(set(df["test group"].dropna()) - {"ad", "psa"}))
print("Unexpected converted values:", sorted(set(df["converted"].dropna()) - {True, False}))
print("Rows with total ads less than 0:", (df["total ads"] < 0).sum())
print("Rows with missing user IDs:", df["user id"].isna().sum())

# Data Cleaning

The raw DataFrame is copied before the unnecessary index column is removed and the column names are made easier to use.

In [ ]:
raw_df = df.copy(deep=True)
original_shape = raw_df.shape
original_group_counts = raw_df["test group"].value_counts().sort_index()
original_conversion_total = raw_df["converted"].sum()

cleaned_df = raw_df.drop(columns=["Unnamed: 0"]).rename(
    columns={
        "user id": "user_id",
        "test group": "test_group",
        "total ads": "total_ads",
        "most ads day": "most_ads_day",
        "most ads hour": "most_ads_hour",
    }
)

valid_days = {
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
}

assert set(cleaned_df["test_group"]) == {"ad", "psa"}
assert cleaned_df["converted"].map(lambda value: isinstance(value, bool)).all()
assert cleaned_df["user_id"].notna().all()
assert cleaned_df["user_id"].is_unique
assert (cleaned_df["total_ads"] >= 0).all()
assert cleaned_df["most_ads_hour"].between(0, 23).all()
assert set(cleaned_df["most_ads_day"]) <= valid_days
assert not cleaned_df.isna().any().any()

print("Validation passed: data checks succeeded.")
print("Cleaned shape:", cleaned_df.shape)
print("Columns removed: ['Unnamed: 0']")
print("Columns renamed: user id, test group, total ads, most ads day, most ads hour")

In [ ]:
project_root = csv_path.parents[2]
cleaned_path = project_root / "data" / "cleaned" / "marketing_ab_cleaned.csv"

cleaned_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(cleaned_path, index=False)
reloaded_df = pd.read_csv(cleaned_path)

assert reloaded_df.shape == (588101, 6)
assert reloaded_df["user_id"].is_unique
assert reloaded_df["test_group"].value_counts().sort_index().equals(original_group_counts)
assert reloaded_df["converted"].sum() == original_conversion_total

comparison_df = reloaded_df.copy()
comparison_df["converted"] = comparison_df["converted"].astype(bool)
pd.testing.assert_frame_equal(
    cleaned_df.reset_index(drop=True),
    comparison_df.reset_index(drop=True),
    check_dtype=False,
)

print(f"Saved and verified: {cleaned_path}")
print("Original shape:", original_shape)
print("Exported shape:", reloaded_df.shape)
print("Original group counts:")
print(original_group_counts)
print("Exported group counts:")
print(reloaded_df["test_group"].value_counts().sort_index())
print("Original conversions:", original_conversion_total)
print("Exported conversions:", reloaded_df["converted"].sum())
print("Exported values match cleaned DataFrame: True")